In [1]:
# ---- site discovery: the best Earth places for every Moon and Mars region (writes export/best_sites.json)
# stage 1  screen all land on a 32 km grid with the 32 km scale only (cheap), after a surface pre-check
# stage 2  score in full (same as a click) the 4 km blocks inside the best cells, plus known geological features
# stage 3  shift the best blocks by 2 km each way and keep the best cut, so the feature sits in the middle
# stage 4  pick the top places per region, spaced apart, name them, check them, write the file
# every stage appends to a csv in OUT as it goes, so a rerun picks up where it stopped.
# run:  cd ml && caffeinate -i jupyter execute --inplace --timeout=-1 discover.ipynb     (progress: tail -f the log below)
import csv, io, json, math, os, sys, time, zipfile
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
import numpy as np, pandas as pd

sys.path.insert(0, "export")
import analog_runtime as rt, analog_readers as rd

SMOKE = os.environ.get("DISCOVER_SMOKE") == "1"          # a few-minute end-to-end test on a tiny grid
MODEL = str(Path("export/model.npz").resolve())
model = rt.load(MODEL)
KEYS = list(model.region_keys)
CFG = dict(
    step_km=32 if not SMOKE else 600,      # stage 1 grid spacing
    lat_range=(-60.0, 84.0),               # GLO-30 / WorldCover / WorldClim land (Antarctica comes in through the known sites)
    precip_max=2500,                       # mm/yr: wetter than this is rainforest, never a usable analog
    surface_min=20,                        # stage 1: cells whose surface score is under this skip the terrain read
    snow_max=0.8,                          # ice sheets: nothing to match, dropped from the picks
    cells_per_region=120 if not SMOKE else 6, cell_spacing_km=64,     # stage 1 -> 2: best cells per region, spaced
    block_offsets_km=(-12, -4, 4, 12),     # stage 2: a 4 x 4 pattern of 4 km blocks inside each chosen cell
    feature_offsets_km=((0, 0), (4, 0), (-4, 0), (0, 4), (0, -4)),    # stage 2: around each known feature
    max_features=None if not SMOKE else 12,
    refine_per_region=50 if not SMOKE else 4, refine_spacing_km=20, shift_km=2,     # stage 3
    best_per_region=25 if not SMOKE else 5, best_spacing_km=50,       # stage 4
    known_km=25,                           # a pick this close to a verified analog site is flagged "known"
    pick_surface_min=40,                   # stage 4 ranks by overall among places with at least this surface score (v3.0 scale)
    pick_spec_max=0,                       # ... whose region is their best region in that world (v3.0: regions differ by a few points, so 3 let one place into most lists)
    pick_per_country=4,                    # ... at most this many per country in one list
    landform_per_list=15,                  # validated landform lists
    workers=int(os.environ.get("DISCOVER_WORKERS", 32)),   # mostly waiting on the network, so more than cores
)
OUT = Path("../data/cache") / f"discover_{model.version}{'_smoke' if SMOKE else ''}"
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "discover.log"

def log(msg):
    line = f"{time.strftime('%H:%M:%S')} {msg}"
    print(line, flush=True)
    with open(LOG, "a") as f:
        f.write(line + "\n")

log(f"model {model.version}, {len(KEYS)} regions, out {OUT.resolve()}, smoke={SMOKE}")

16:41:18 model regions-3.0-v1, 8 regions, out /Users/victor/Desktop/spaceapps/data/cache/discover_regions-3.0-v1, smoke=False


In [2]:
%%writefile discover_tasks.py
# worker functions for discover.ipynb (in a module so process pools can import them)
import os, sys
from pathlib import Path

os.environ.setdefault("GDAL_HTTP_TIMEOUT", "45")           # a stuck read must not hang a worker all night
os.environ.setdefault("GDAL_HTTP_CONNECTTIMEOUT", "15")
sys.path.insert(0, str(Path(__file__).resolve().parent / "export"))

import numpy as np
import rasterio, rasterio.enums, rasterio.windows
from scipy.special import expit

import analog_runtime as rt
import analog_readers as rd

_M = {}


def get_model(path):
    m = _M.get(path)
    if m is None:
        m = _M[path] = rt.load(path)
        rd.glo30_tiles(Path(__file__).resolve().parent / "datasets/glo30_tileList.txt")   # local copy of the tile list
    return m


def surface_score(m, env):
    # the 0-100 surface score, the same formula as score_click
    G, _ = rt.conditions(m, env)
    return float(min(100.0, 100.0 * 100.0 * expit(G) / m.surface_max))


def coarse_cover(lat, lon, half_m, px=96):
    # rd.worldcover_shares over a wide window, read decimated so it's one small request. None when there's no tile
    dlat = np.degrees(half_m / rt.RADIUS_M["earth"])
    dlon = dlat / max(np.cos(np.radians(lat)), 1e-3)
    la0, lo0 = int(np.floor(lat / 3) * 3), int(np.floor(lon / 3) * 3)
    name = f"ESA_WorldCover_10m_2021_v200_{'N' if la0 >= 0 else 'S'}{abs(la0):02d}{'E' if lo0 >= 0 else 'W'}{abs(lo0):03d}_Map.tif"
    try:
        with rasterio.Env(**rd.GDAL_ENV), rasterio.open(f"/vsicurl/{rd.WORLDCOVER}/{name}") as ds:
            w = rasterio.windows.from_bounds(lon - dlon, lat - dlat, lon + dlon, lat + dlat, ds.transform)
            w = w.round_offsets().round_lengths().intersection(rasterio.windows.Window(0, 0, ds.width, ds.height))
            a = ds.read(1, window=w, out_shape=(max(1, min(px, int(w.height))), max(1, min(px, int(w.width)))),
                        resampling=rasterio.enums.Resampling.nearest)
    except Exception:
        return None
    a = a[a > 0]
    if a.size == 0:
        return None
    s = {n: float(np.mean(a == k)) for k, n in rd.WC.items()}
    return dict(tree=s["tree"] + s["mangrove"], crop=s["crop"], built=s["built"], water=s["water"] + s["wetland"],
                snow=s["snow"], bare=s["bare"], sparse=s["shrub"] + s["grass"] + s["moss"])


def _key(lat, lon):
    return f"{lat:.4f},{lon:.4f}"


def screen_task(args):
    # stage 1: one 32 km cell. surface first (one small read); terrain only if the surface could pass
    path, lat, lon, surface_min = args
    row = dict(key=_key(lat, lon), lat=lat, lon=lon, status="ok")
    for attempt in range(2):
        try:
            m = get_model(path)
            env = coarse_cover(lat, lon, 16000.0) or {n: np.nan for n in rd.WC_NAMES}
            env["precip"] = rt.precip_at(m, lat, lon)
            row.update(surface=surface_score(m, env), snow=env.get("snow", np.nan), precip=env["precip"])
            if row["surface"] < surface_min:
                row["status"] = "low_surface"
                return row
            dem = rd.read_glo30_window(lat, lon, rt.SCALES["regional"]["res"])
            if np.isnan(dem).mean() > 0.4 or rd.flat_fraction(np.nan_to_num(dem, nan=-1e9)) > 0.5:
                row["status"] = "no_terrain"
                return row
            emb = rt.embed(m, {"regional": dem}, rt.RADIUS_M["earth"])
            # v3.0 models: the terrain head on the 32 km measurements alone (the 4 km ones read as typical)
            head = rt.terrain_head(m, emb) if hasattr(rt, "terrain_head") else None
            for w in rt.WORLDS:
                for reg in (rt.region_scores(m, emb, w, n_best=1, head=head) if head is not None else rt.region_scores(m, emb, w, n_best=1)):
                    row[reg["region"]] = reg["score"]
            return row
        except Exception as e:
            row["status"] = f"error: {type(e).__name__}: {str(e)[:80]}"
    return row


def full_task(args):
    # stages 2 and 3: one 4 km block scored exactly like a click
    path, lat, lon, source = args
    row = dict(key=_key(lat, lon), lat=lat, lon=lon, status="ok", source=source)
    for attempt in range(2):
        try:
            m = get_model(path)
            dems, env = rd.fetch_earth(m, lat, lon)
            r = rt.score_click(m, dems, env, n_best=1)
            row.update(status="ok", surface=r["conditions"]["score"], snow=env.get("snow", np.nan), ood=int(r["ood"]),
                       scales="+".join(r["scales_used"]))
            for W in r["worlds"].values():
                for reg in W["regions"]:
                    row[reg["region"]] = reg["score"]
                    lf = reg.get("landform")
                    if lf:
                        row["lf_" + reg["region"]] = f"{lf['tag']}:{lf['score']:.1f}:{int(lf['validated'])}"
            return row
        except ValueError as e:
            if str(e).startswith("no_terrain"):
                row["status"] = "no_terrain"
                return row
            row["status"] = f"error: ValueError: {str(e)[:80]}"
        except Exception as e:
            row["status"] = f"error: {type(e).__name__}: {str(e)[:80]}"
    return row

Overwriting discover_tasks.py


In [3]:
# ---- shared helpers: the resumable stage runner, distances, spaced picks
import importlib, discover_tasks
importlib.reload(discover_tasks)
R_KM = rt.RADIUS_M["earth"] / 1000.0
S1_COLS = ["key", "lat", "lon", "status", "surface", "snow", "precip", *KEYS]
FULL_COLS = ["key", "lat", "lon", "status", "source", "surface", "snow", "ood", "scales", *KEYS, *[f"lf_{k}" for k in KEYS]]

def read_stage(name):
    p = OUT / f"{name}.csv"
    if not p.exists() or p.stat().st_size == 0:
        return pd.DataFrame(columns=S1_COLS if name == "stage1" else FULL_COLS)
    return pd.read_csv(p, on_bad_lines="skip").drop_duplicates("key", keep="last").reset_index(drop=True)

def run_stage(name, task, jobs, cols):
    # jobs: {key: args}. appends one csv row per finished job; a rerun (or a crashed pool) skips what's already there
    path = OUT / f"{name}.csv"
    t0, n_new = time.time(), 0
    for restart in range(30):
        done = set(read_stage(name)["key"])
        todo = [a for k, a in jobs.items() if k not in done]
        if restart == 0:
            log(f"{name}: {len(jobs)} jobs, {len(jobs) - len(todo)} already done, {len(todo)} to do")
        if not todo:
            break
        try:
            with ProcessPoolExecutor(CFG["workers"]) as ex, open(path, "a", newline="") as f:
                wr = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
                if f.tell() == 0:
                    wr.writeheader()
                t1 = time.time()
                for n_pass, row in enumerate(ex.map(task, todo, chunksize=2), 1):
                    wr.writerow(row)
                    n_new += 1
                    if n_pass % 50 == 0:
                        f.flush()
                    if n_pass % 1000 == 0:
                        rate = n_pass / (time.time() - t1)
                        log(f"{name}: {n_new} new, {rate:.1f}/s, about {(len(todo) - n_pass) / max(rate, 1e-9) / 60:.0f} min left")
        except Exception as e:                       # a dead worker breaks the pool: start a new one and carry on
            log(f"{name}: pool stopped ({type(e).__name__}: {str(e)[:100]}), restarting")
            time.sleep(5)
    df = read_stage(name)
    log(f"{name}: finished, {len(df)} rows ({dict(df.status.str.slice(0, 12).value_counts())}), {n_new} new in {(time.time() - t0) / 60:.1f} min")
    return df

def km_between(lat, lon, lats, lons):
    p1, p2 = np.radians(lat), np.radians(np.asarray(lats, float))
    dl = np.radians(np.asarray(lons, float) - lon)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def offset(lat, lon, east_km, north_km):
    la = lat + math.degrees(north_km / R_KM)
    lo = lon + math.degrees(east_km / (R_KM * max(math.cos(math.radians(lat)), 1e-3)))
    return round(la, 4), round((lo + 180.0) % 360.0 - 180.0, 4)

def spaced_top(df, col, k, min_km):
    # the k highest rows by col that are at least min_km from each other
    d = df[np.isfinite(df[col])].sort_values(col, ascending=False)
    keep_lat, keep_lon, idx = [], [], []
    for i, la, lo in zip(d.index, d.lat.values, d.lon.values):
        if keep_lat and km_between(la, lo, keep_lat, keep_lon).min() < min_km:
            continue
        keep_lat.append(la); keep_lon.append(lo); idx.append(i)
        if len(idx) >= k:
            break
    return d.loc[idx]

def add_overall(df):
    # overall per region = sqrt(terrain x surface), the number a click shows. ice sheets are left out
    df = df[df.status == "ok"].reset_index(drop=True).copy()      # a clean index: spaced_top picks rows by label
    ok = ~(df.snow.astype(float) > CFG["snow_max"])
    for k in KEYS:
        # the same overall a click shows (v2.1: landscape x (surface/100)^beta; older models sqrt(landscape x surface))
        o = [rt.overall_of(model, t, s) for t, s in zip(df[k].astype(float), df.surface.astype(float))] if hasattr(rt, "overall_of") \
            else np.sqrt(np.clip(df[k].astype(float), 0, None) * df.surface.astype(float))
        df["o_" + k] = np.where(ok, o, np.nan)
    return df

# verified reference lists (all local files)
SITES = pd.read_csv("analog_sites.csv")
NEG = pd.read_csv("locations/earth_negative.csv")
CRATERS = pd.read_csv("earth_craters.csv")
gvp = json.load(open("datasets/gvp_holocene.json"))["features"]
VOLC = pd.DataFrame([dict(name=f["properties"]["Volcano_Name"], lon=f["geometry"]["coordinates"][0], lat=f["geometry"]["coordinates"][1])
                     for f in gvp if f.get("geometry")])
log(f"reference lists: {len(SITES)} analog sites, {len(VOLC)} GVP volcanoes, {len(CRATERS)} craters, {len(NEG)} negative places")

16:41:18 reference lists: 114 analog sites, 1214 GVP volcanoes, 38 craters, 16 negative places


In [4]:
# ---- stage 1: screen all land on the grid (surface pre-check, then the 32 km scale only)
def land_grid(step_km):
    g = model.precip
    dlat = math.degrees(step_km / R_KM)
    pts = []
    for lat in np.arange(CFG["lat_range"][0] + dlat / 2, CFG["lat_range"][1], dlat):
        dlon = dlat / max(math.cos(math.radians(lat)), 1e-3)
        lons = np.arange(-180 + dlon / 2, 180, dlon)
        i = min(max(int((90.0 - lat) / 180.0 * g.shape[0]), 0), g.shape[0] - 1)
        j = np.clip((((lons + 180.0) % 360.0) / 360.0 * g.shape[1]).astype(int), 0, g.shape[1] - 1)
        p = g[i, j].astype(float)
        keep = (p < 65535) & (p <= CFG["precip_max"])          # WorldClim land, and not rainforest
        pts += [(round(float(lat), 4), round(float(lo), 4)) for lo in lons[keep]]
    return pts

grid = land_grid(CFG["step_km"])
log(f"stage 1 grid: {len(grid)} land cells at {CFG['step_km']} km (precip <= {CFG['precip_max']} mm)")
S1 = run_stage("stage1", discover_tasks.screen_task,
               {f"{la:.4f},{lo:.4f}": (MODEL, la, lo, CFG["surface_min"]) for la, lo in grid}, S1_COLS)

16:41:18 stage 1 grid: 131957 land cells at 32 km (precip <= 2500 mm)


16:41:18 stage1: 131957 jobs, 131957 already done, 0 to do


16:41:18 stage1: finished, 131957 rows ({'low_surface': np.int64(73291), 'ok': np.int64(58211), 'no_terrain': np.int64(455)}), 0 new in 0.0 min

In [5]:
# ---- stage 1 results: best cells per region, and does the screen keep the verified analog sites?
S1o = add_overall(S1)
log(f"stage 1: {len(S1)} cells, {len(S1o)} scored, {int((S1.status == 'low_surface').sum())} skipped on surface, "
    f"{int(S1.status.str.startswith('error').sum())} errors")
chosen = {}
for k in KEYS:
    chosen[k] = spaced_top(S1o, "o_" + k, CFG["cells_per_region"], CFG["cell_spacing_km"])
CELLS = pd.concat(chosen.values()).drop_duplicates("key").reset_index(drop=True)
log(f"stage 1 -> 2: {len(CELLS)} cells chosen ({CFG['cells_per_region']} per region, {CFG['cell_spacing_km']} km apart)")

def screen_check(df, label):
    # each place's 32 km cell: its best-region overall as a percentile of all screened land (cells skipped on surface count as 0)
    rows = []
    for _, s in df.iterrows():
        if not len(S1):
            break
        d = km_between(s.lat, s.lon, S1.lat.values, S1.lon.values)
        j = int(np.argmin(d))
        if d[j] > CFG["step_km"]:
            continue                                   # off the grid (Antarctica, small islands)
        c = S1.iloc[j]
        world = str(s.get("world", "")) if "world" in df.columns else ""
        ks = [k for k in KEYS if k.startswith(world + ":")] or KEYS
        best, pct = 0.0, 0.0
        if c.status == "ok" and c.key in S1o.key.values:
            co = S1o[S1o.key == c.key].iloc[0]
            for k in ks:
                v = co["o_" + k]
                if np.isfinite(v) and v >= best:
                    best = float(v)
                    pct = float((S1o["o_" + k].fillna(0) < v).sum() + (len(S1) - len(S1o))) / len(S1)
        rows.append(dict(name=s["name"], status=c.status[:11], overall=round(best, 1), percentile=round(100 * pct, 1),
                         in_chosen=bool(len(CELLS) and km_between(s.lat, s.lon, CELLS.lat.values, CELLS.lon.values).min() <= CFG["step_km"])))
    t = pd.DataFrame(rows)
    if len(t):
        log(f"screen check, {label}: {len(t)} on the grid, median percentile {t.percentile.median():.0f}, "
            f"{(t.percentile >= 75).mean():.0%} in the top quarter, {t.in_chosen.mean():.0%} next to a chosen cell")
    return t

REC = screen_check(SITES[SITES.kind == "terrain"], "verified terrain analog sites (higher is better)")
NEGC = screen_check(NEG, "negative set (lower is better)")
print(REC.sort_values("percentile").to_string(index=False))

16:41:18 stage 1: 131957 cells, 58211 scored, 73291 skipped on surface, 0 errors


16:41:18 stage 1 -> 2: 476 cells chosen (120 per region, 64 km apart)


16:41:19 screen check, verified terrain analog sites (higher is better): 77 on the grid, median percentile 94, 73% in the top quarter, 8% next to a chosen cell


16:41:19 screen check, negative set (lower is better): 15 on the grid, median percentile 0, 0% in the top quarter, 0% next to a chosen cell


                                  name      status  overall  percentile  in_chosen
        Channeled Scablands, Dry Falls low_surface      0.0         0.0      False
                             Reykjanes low_surface      0.0         0.0      False
                       Nördlinger Ries low_surface      0.0         0.0      False
                      Newberry Volcano low_surface      0.0         0.0      False
                            Mount Etna low_surface      0.0         0.0      False
                  Lanzarote, Timanfaya low_surface      0.0         0.0      False
                        Valles Caldera low_surface      0.0         0.0      False
                     Turpan Depression low_surface      0.0         0.0      False
                     Bletterbach gorge low_surface      0.0         0.0      False
                          Lonar crater low_surface      0.0         0.0      False
                         Sudbury Basin low_surface      0.0         0.0      False
    

In [6]:
# ---- stage 2: full scoring of the blocks inside the chosen cells, plus known geological features
jobs = {}
def add_job(la, lo, source):
    la, lo = round(float(la), 4), round(float(lo), 4)
    jobs.setdefault(f"{la:.4f},{lo:.4f}", (MODEL, la, lo, source))

for _, c in CELLS.iterrows():
    for dy in CFG["block_offsets_km"]:
        for dx in CFG["block_offsets_km"]:
            add_job(*offset(c.lat, c.lon, dx, dy), "screen")
n_screen = len(jobs)
feats = [("analog", SITES[SITES.kind == "terrain"]), ("crater", CRATERS), ("volcano", VOLC)]
for label, df in feats:
    df = df if CFG["max_features"] is None else df.head(CFG["max_features"])
    for _, s in df.iterrows():
        for dx, dy in CFG["feature_offsets_km"]:
            add_job(*offset(s.lat, s.lon, dx, dy), f"{label}:{s['name']}")
log(f"stage 2: {n_screen} blocks from the screen + {len(jobs) - n_screen} around known features")
S2 = run_stage("stage2", discover_tasks.full_task, jobs, FULL_COLS)

16:41:19 stage 2: 7616 blocks from the screen + 6645 around known features


16:41:19 stage2: 14261 jobs, 14261 already done, 0 to do


16:41:19 stage2: finished, 14261 rows ({'ok': np.int64(12702), 'no_terrain': np.int64(1559)}), 0 new in 0.0 min


In [7]:
# ---- stage 3: centring. shift the best blocks 2 km each way and keep whichever cut scores best
S2o = add_overall(S2)
jobs3 = {}
for k in KEYS:
    for _, b in spaced_top(S2o, "o_" + k, CFG["refine_per_region"], CFG["refine_spacing_km"]).iterrows():
        for dy in (-CFG["shift_km"], 0, CFG["shift_km"]):
            for dx in (-CFG["shift_km"], 0, CFG["shift_km"]):
                if dx or dy:
                    la, lo = offset(b.lat, b.lon, dx, dy)
                    jobs3.setdefault(f"{la:.4f},{lo:.4f}", (MODEL, la, lo, f"shift of {b.source}"))
S3 = run_stage("stage3", discover_tasks.full_task, jobs3, FULL_COLS)
ALL = add_overall(pd.concat([S2, S3]).drop_duplicates("key", keep="last"))
ALL.to_csv(OUT / "all_scored.csv", index=False)
log(f"stage 3: {len(ALL)} fully scored blocks in all")

16:41:19 stage3: 884 jobs, 884 already done, 0 to do


16:41:19 stage3: finished, 884 rows ({'ok': np.int64(884)}), 0 new in 0.0 min


16:41:19 stage 3: 13586 fully scored blocks in all


In [8]:
# ---- stage 4: pick, name, check, write
def load_places():
    # GeoNames cities1000 (CC BY 4.0), downloaded once, for readable names. without it names are just coordinates
    p = Path("datasets/gazetteer/cities1000.zip")
    try:
        if not p.exists():
            p.write_bytes(rd.http("https://download.geonames.org/export/dump/cities1000.zip", timeout=120))
        with zipfile.ZipFile(p) as z:
            t = pd.read_csv(io.BytesIO(z.read("cities1000.txt")), sep="\t", header=None, usecols=[1, 4, 5, 8, 14],
                            names=["name", "lat", "lon", "cc", "pop"], keep_default_na=False, na_values=[""])
        return t[t["pop"] >= 5000].reset_index(drop=True)
    except Exception as e:
        log(f"no place names ({type(e).__name__}: {str(e)[:80]}), using coordinates")
        return None

PLACES = load_places()
COMPASS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]

def place_name(lat, lon, source):
    # a known feature if the block came from one, else "42 km NE of Town, CC", else the coordinates
    src = str(source).replace("shift of ", "")
    if ":" in src and not src.startswith("screen"):
        return src.split(":", 1)[1]
    if PLACES is not None:
        d = km_between(lat, lon, PLACES.lat.values, PLACES.lon.values)
        j = int(np.argmin(d))
        t = PLACES.iloc[j]
        if d[j] < 8:
            return f"{t['name']}, {t.cc}"
        y = math.sin(math.radians(lon - t.lon)) * math.cos(math.radians(lat))
        x = math.cos(math.radians(t.lat)) * math.sin(math.radians(lat)) - math.sin(math.radians(t.lat)) * math.cos(math.radians(lat)) * math.cos(math.radians(lon - t.lon))
        return f"{d[j]:.0f} km {COMPASS[int(round(math.degrees(math.atan2(y, x)) / 45)) % 8]} of {t['name']}, {t.cc}"
    return f"{abs(lat):.2f}°{'N' if lat >= 0 else 'S'}, {abs(lon):.2f}°{'E' if lon >= 0 else 'W'}"

# v2: rank by terrain (surface is near 100 for any bare place, so overall mostly ranked bareness), only where this
# region is (near) the block's best region in its world, spaced 50 km apart and at most 4 per country per list
WORLD_KEYS = {w: [k for k in KEYS if k.startswith(w + ":")] for w in rt.WORLDS}
for w, ks in WORLD_KEYS.items():
    best_in_world = ALL[ks].astype(float).max(axis=1)
    for k in ks:
        ALL["spec_" + k] = ALL[k].astype(float) - best_in_world          # 0 = this is the block's best region
ELIG = ALL[ALL.surface.astype(float) >= CFG["pick_surface_min"]].reset_index(drop=True)
NEAR = {}

def country(lat, lon):
    if PLACES is None:
        return ""
    return str(PLACES.cc.values[int(np.argmin(km_between(lat, lon, PLACES.lat.values, PLACES.lon.values)))])

def pick(df, col, k, min_km, per_country):
    # like spaced_top, plus a cap per country
    d = df[np.isfinite(df[col])].sort_values(col, ascending=False)
    keep_lat, keep_lon, idx, per = [], [], [], {}
    for i, la, lo in zip(d.index, d.lat.values, d.lon.values):
        if keep_lat and km_between(la, lo, keep_lat, keep_lon).min() < min_km:
            continue
        cc = country(la, lo)
        if per.get(cc, 0) >= per_country:
            continue
        per[cc] = per.get(cc, 0) + 1
        keep_lat.append(la); keep_lon.append(lo); idx.append(i)
        if len(idx) >= k:
            break
    return d.loc[idx]

def site_row(b, k, score_col):
    d = km_between(b.lat, b.lon, SITES.lat.values, SITES.lon.values)
    j = int(np.argmin(d))
    known = bool(d[j] <= CFG["known_km"])
    lf = str(b.get("lf_" + k, ""))
    src = str(b.source).replace("shift of ", "")
    return dict(name=place_name(b.lat, b.lon, b.source), lat=float(b.lat), lon=float(b.lon), country=country(b.lat, b.lon),
                tile_id=f"earth:p:{b.lat:.4f}:{b.lon:.4f}", score=round(float(b[score_col]), 1),
                terrain=round(float(b[k]), 1), surface=round(float(b.surface), 1),
                overall=round(float(b["o_" + k]), 1) if np.isfinite(b["o_" + k]) else None,     # NaN on snow-covered blocks
                lead=round(float(-max(v for kk, v in ((kk, b[kk]) for kk in WORLD_KEYS[k.split(":")[0]] if kk != k)) + b[k]), 1),
                known=known, known_site=str(SITES.name.values[j]) if known else None,
                found_by="screen" if src.startswith("screen") else src.split(":")[0],
                landform=lf if lf and lf != "nan" else None)

worlds, flat = {w: [] for w in rt.WORLDS}, []
for r, k in enumerate(KEYS):
    cand = ELIG[ELIG["spec_" + k] >= -CFG["pick_spec_max"]]
    # v3.0: rank by overall, the number a click shows (terrain and surface are both trained on verified sites now)
    top = pick(cand, "o_" + k, CFG["best_per_region"], CFG["best_spacing_km"], CFG["pick_per_country"])
    sites = [site_row(b, k, "o_" + k) for _, b in top.iterrows()]
    flat += [dict(region=k, **s) for s in sites]
    worlds[str(model.region_world[r])].append(dict(region=k, title=str(model.region_title[r]), n_candidates=int(len(cand)), sites=sites))

# validated landform lists: blocks whose shown landform for that region is this one, ranked by the landform score
landforms = []
for i in range(len(model.lf_keys)):
    if not model.lf_valid[i]:
        continue
    k, tag = str(model.region_keys[model.lf_region[i]]), str(model.lf_tag[i])
    col = "lf_" + k
    parts = ELIG[col].astype(str).str.split(":", expand=True) if col in ELIG else None
    if parts is None or parts.shape[1] < 3:
        continue
    cand = ELIG[(parts[0] == tag) & (parts[2] == "1")].copy()
    cand["lfscore"] = parts.loc[cand.index, 1].astype(float)
    top = pick(cand, "lfscore", CFG["landform_per_list"], CFG["best_spacing_km"], CFG["pick_per_country"])
    sites = []
    for _, b in top.iterrows():
        s_ = site_row(b, k, "lfscore")
        sites.append(s_)
    landforms.append(dict(region=k, region_title=str(model.region_title[r if False else model.lf_region[i]]),
                          tag=tag, label=str(model.lf_label[i]), sites=sites))
    log(f"  landform list {k} | {tag}: {len(sites)} sites from {len(cand)} candidates")

BEST = pd.DataFrame(flat)
out = dict(status="computed", model_version=model.version, generated=time.strftime("%Y-%m-%d"),
           note=f"top Earth blocks per region by overall score, among places with a surface score of at least {CFG['pick_surface_min']} "
                f"whose region is within {CFG['pick_spec_max']} points of their best region, at least {CFG['best_spacing_km']} km apart "
                f"and at most {CFG['pick_per_country']} per country. place names from GeoNames (CC BY 4.0)",
           worlds=worlds, landforms=landforms)
if not SMOKE:
    json.dump(out, open("export/best_sites.json", "w"), indent=1, ensure_ascii=False, allow_nan=False)   # NaN isn't JSON
    BEST.to_csv("export/best_sites.csv", index=False)
json.dump(out, open(OUT / "best_sites.json", "w"), indent=1, ensure_ascii=False)
log(f"stage 4: wrote {len(BEST)} sites over {len(KEYS)} regions" + ("" if not SMOKE else " (smoke: only in OUT)"))

# checks
if len(BEST):
    neg_near = [n for _, n in NEG.iterrows() if km_between(n.lat, n.lon, BEST.lat.values, BEST.lon.values).min() < 40]
    log(f"negative set: {len(neg_near)} of {len(NEG)} places within 40 km of a pick -> {'PASS' if not neg_near else 'FAIL: ' + ', '.join(n['name'] for n in neg_near)}")
    log(f"picks: {BEST.known.mean():.0%} next to a verified analog site; found by {dict(BEST.found_by.value_counts())}")
    for k in KEYS:
        b = BEST[BEST.region == k]
        log(f"  {k:18s} n={len(b):2d}  overall {b.score.max():.0f}..{b.score.min():.0f}  surface >= {b.surface.min():.0f}  known {int(b.known.sum()):2d}  "
            f"countries {b.country.nunique():2d}  top: {b.name.iloc[0] if len(b) else '-'}")
    log(f"country share: {dict(BEST.country.value_counts().head(8))}")
    print(BEST.to_string(index=False, max_colwidth=40))
log("done")

16:41:20   landform list moon:maria | crater_chain: 15 sites from 3870 candidates


16:41:20   landform list moon:maria | wrinkle_ridge: 15 sites from 6410 candidates


16:41:20   landform list moon:highlands | sinuous_rille: 15 sites from 3314 candidates


16:41:20   landform list moon:highlands | small_crater: 15 sites from 2669 candidates


16:41:20   landform list moon:highlands | wrinkle_ridge: 15 sites from 4297 candidates


16:41:20   landform list mars:highlands | crater_rim: 15 sites from 5652 candidates


... (22 outputs left out)

16:41:21   mars:volcanic      n=25  overall 96..84  surface >= 61  known  2  countries 11  top: 40 km SE of Hōlualoa, US


16:41:21   mars:outflow       n=25  overall 96..92  surface >= 71  known  4  countries 13  top: 129 km E of Diego de Almagro, CL


16:41:21 country share: {'CN': np.int64(28), 'CL': np.int64(25), 'AR': np.int64(23), 'US': np.int64(21), 'MN': np.int64(11), 'ET': np.int64(9), 'IR': np.int64(7), 'TD': np.int64(6)}


         region                                     name      lat       lon country                   tile_id  score  terrain  surface  overall  lead  known            known_site found_by                landform
     moon:maria 70 km SW of Sandaoling Lutiankuang Wu...  42.7025   92.1320      CN   earth:p:42.7025:92.1320   95.5     98.0     85.7     95.5   1.8  False                   NaN   screen    wrinkle_ridge:98.0:1
     moon:maria                                  Mat Ala  13.0880   41.1425      ET   earth:p:13.0880:41.1425   95.0     97.4     85.8     95.0   1.7  False                   NaN  volcano    wrinkle_ridge:97.8:1
     moon:maria                     95 km SW of Khūr, IR  33.0798   54.4973      IR   earth:p:33.0798:54.4973   94.1     96.3     85.8     94.1   1.3  False                   NaN   screen    wrinkle_ridge:97.1:1
     moon:maria                                 Dama Ali  11.2260   41.6483      ET   earth:p:11.2260:41.6483   93.6     95.8     85.2     93.6   0.1  F